In [77]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import seaborn as sns
from mlxtend.plotting import plot_decision_regions
import os

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["PYTHONHASHSEED"] = "0"

# 2. Set non-TensorFlow seeds
import random
import numpy as np

random.seed(42)
np.random.seed(42)

# 3. Import TensorFlow and set its seed last
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten,BatchNormalization

tf.random.set_seed(42)
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam
import pandas as pd

In [78]:
df=pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [79]:
df.shape

(768, 9)

In [80]:
df.duplicated().sum()

np.int64(0)

In [81]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [82]:
X=df.iloc[:,:-1]
y=df.iloc[:,-1]

In [83]:
y.head()

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [84]:
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split 
X_train, X_test , y_train , y_test=train_test_split(X,y , test_size=0.2 , random_state=42)

In [85]:
scaler= StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)

In [86]:
X_test_scaled=scaler.transform(X_test)

In [87]:
model=Sequential([
    Dense(units=32,activation='relu'),
    Dense(units=1,activation='sigmoid')]
)

In [88]:
model.compile(optimizer=Adam(learning_rate=0.1),metrics=['accuracy'],loss='binary_crossentropy')

In [89]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [90]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',         
    patience=10,                
    restore_best_weights=True
)
model.fit(X_train_scaled,y_train,batch_size=32,epochs=100,callbacks=[early_stop],validation_split=0.2,verbose=1)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6945 - loss: 0.5623 - val_accuracy: 0.7236 - val_loss: 0.5249
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7475 - loss: 0.4976 - val_accuracy: 0.7236 - val_loss: 0.4721
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7739 - loss: 0.4700 - val_accuracy: 0.7480 - val_loss: 0.5076
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7719 - loss: 0.4686 - val_accuracy: 0.7480 - val_loss: 0.5031
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7637 - loss: 0.4636 - val_accuracy: 0.7480 - val_loss: 0.4721
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7984 - loss: 0.4250 - val_accuracy: 0.7561 - val_loss: 0.4984
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7963 - loss: 0.4301 - val_accuracy: 0.7642 - val_loss: 0.4931
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7923 - loss: 0.4256 - val_accuracy: 0.7724 - 

In [91]:
import keras_tuner as kt

In [92]:
def model_build(hp):
    model=Sequential()

    for i in range(hp.Int('num_layer' , min_value=1, max_value=4)):   
        
        
        model.add(
            Dense(
                units=hp.Int('unit' + str(i) , min_value=8 , max_value=128 , step=8), 
                activation=hp.Choice('activation' + str(i) , values=['relu','tanh'])
            )
        )
        model.add(
            Dropout(
                hp.Float("dropout"+str(i),0.1,0.4,step=0.1)
            )
        )
        
        
    model.add(Dense(units=1, activation='sigmoid'))
    
    optimizer_name = hp.Choice(
        "optimizer",
        ["adam", "rmsprop"]
    )
    
    learning_rate = hp.Choice(
        "learning_rate",
        [0.01 , 0.1 , 0.5]
    )
    
    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate)
    
    else:
        optimizer = tf.keras.optimizers.RMSprop(learning_rate)

    model.compile(optimizer=optimizer,metrics=['accuracy'],loss='binary_crossentropy')
    return model

In [93]:
tuner=kt.RandomSearch(
    model_build,
    objective='val_accuracy',
    max_trials=20, 
    directory="my_tuner",
    project_name="diabetes"
)

In [94]:
tuner.search(X_train_scaled,
             y_train , 
             validation_split=0.2,
             epochs=50,
             callbacks=[early_stop],
              verbose=1
            )

Trial 20 Complete [00h 00m 07s]
val_accuracy: 0.7235772609710693

Best val_accuracy So Far: 0.7967479825019836
Total elapsed time: 00h 02m 07s


In [95]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'num_layer': 1, 'unit0': 24, 'activation0': 'relu', 'dropout0': 0.4, 'optimizer': 'adam', 'learning_rate': 0.1, 'unit1': 104, 'activation1': 'relu', 'dropout1': 0.2, 'unit2': 32, 'activation2': 'relu', 'dropout2': 0.1, 'unit3': 48, 'activation3': 'relu', 'dropout3': 0.2}


In [96]:
best_model = tuner.get_best_models(num_models=1)[0]

C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [97]:
best_model

<Sequential name=sequential, built=True>

In [98]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',         
    patience=10,                
    restore_best_weights=True
)
best_model.fit(X_train_scaled,y_train,batch_size=32,epochs=100,callbacks=[early_stop],validation_split=0.2,verbose=1)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7760 - loss: 0.5181 - val_accuracy: 0.6992 - val_loss: 0.5156
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7515 - loss: 0.4729 - val_accuracy: 0.7154 - val_loss: 0.5311
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7617 - loss: 0.4858 - val_accuracy: 0.7317 - val_loss: 0.4955
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7556 - loss: 0.4918 - val_accuracy: 0.7724 - val_loss: 0.4578
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7536 - loss: 0.4941 - val_accuracy: 0.7398 - val_loss: 0.4913
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7556 - loss: 0.4761 - val_accuracy: 0.6829 - val_loss: 0.4889
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7373 - loss: 0.4784 - val_accuracy: 0.7236 - val_loss: 0.4767
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7454 - loss: 0.4612 - val_accuracy: 0.7236 - 